# ML Discovery: Baseline Model Development

## Strategic Objective
- **Objective:** Establish a robust 'Endogenous Baseline' model using internal market data.
- **Methodology:** Use internal signals (Volume, Volatility, Divergence) as a performance floor before integrating external macro variables.
- **Goal:** Identify predictive power of 'Volume Shocks' for T+30 price returns.

In [1]:
import pandas as pd
import sys
import os
from sklearn.ensemble import RandomForestClassifier

# Fix path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.data.bq_connector import BigQueryConnector

connector = BigQueryConnector()
df = connector.fetch_historical_trends()
df['ds'] = pd.to_datetime(df['ds'])

# Feature Engineering
df['price_vol_corr'] = df.groupby('id', group_keys=False).apply(lambda x: x['price'].rolling(30).corr(x['total_volume']))
df['volatility_30d'] = df.groupby('id')['daily_variation'].rolling(30).std().reset_index(0, drop=True)
df['zscore_30d'] = df.groupby('id', group_keys=False).apply(lambda x: (x['total_volume'] - x['total_volume'].rolling(30).mean()) / x['total_volume'].rolling(30).std())
df['target'] = (df.groupby('id')['price'].pct_change(30).shift(-30) > 0.05).astype(int)

df = df.dropna()
print("Features computed.")

[INFO] Adjusted credentials path for notebook context: ../auth/credentials.json
Executing query on: gcp-data-plataform-hub.analytics_data_gold.crypto_historical_trends


/Users/charlie/ai_projects/portafolio/crypto-ml-predictor/venv/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Features computed.


/var/folders/j9/mtv0lz0x6cb1bbktm7dcl8_m0000gn/T/ipykernel_92127/2918361962.py:16: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df['price_vol_corr'] = df.groupby('id', group_keys=False).apply(lambda x: x['price'].rolling(30).corr(x['total_volume']))
/var/folders/j9/mtv0lz0x6cb1bbktm7dcl8_m0000gn/T/ipykernel_92127/2918361962.py:18: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df['zscore_30d'] = df.groupby('id', grou

In [2]:
features = ['zscore_30d', 'price_vol_corr', 'volatility_30d', 'market_cap_dominance']
X = df[features]
y = df['target']

rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf.fit(X, y)
print(pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False))

price_vol_corr          0.344643
volatility_30d          0.281721
market_cap_dominance    0.195164
zscore_30d              0.178472
dtype: float64
